# Barcelona — Listings Analysis
**Input:** `../../Data/interim/barcelona_listings_clean.parquet`  
**Prerequisite:** run `01_cleaning.ipynb` first.

Covers univariate, bivariate, and multivariate analysis of cleaned listings.

In [ ]:
import sys
sys.path.insert(0, "../..")

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import f_oneway, ttest_ind
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from airbnb_iip.data.cleaning import (
    clean_listings, missing_report,
    standardize_property_type, rate_bucket,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 200)

## 1 · Load cleaned listings

In [ ]:
l_df = pd.read_parquet("../../Data/interim/barcelona_listings_clean.parquet")
print(f"Shape: {l_df.shape}")
l_df.head()

## 2 · Univariate analysis

### 2.1 Numerical distributions

In [ ]:
num_vars = [
    "price", "number_of_reviews", "review_scores_rating",
    "accommodates", "calculated_host_listings_count",
]
df_num = l_df[num_vars].dropna()

stats = df_num.describe().T
stats["skewness"] = df_num.skew()
display(stats)

for col in num_vars:
    fig = make_subplots(
        rows=1, cols=2, column_widths=[0.6, 0.4],
        subplot_titles=(f"Histogram — {col}", f"Boxplot — {col}"),
    )
    fig.add_trace(
        go.Histogram(x=df_num[col], nbinsx=40, histnorm="percent", name="Hist"),
        row=1, col=1,
    )
    fig.add_trace(
        go.Box(x=df_num[col], boxmean=True, name="Box"),
        row=1, col=2,
    )
    fig.update_layout(title=f"Univariate: {col}", height=380, showlegend=False)
    fig.show()

### 2.2 Categorical distributions

In [ ]:
cat_vars = [
    "room_type", "property_type_std", "neighbourhood_group_cleansed",
    "host_is_superhost", "instant_bookable", "host_identity_verified",
]
for col in cat_vars:
    counts = l_df[col].value_counts(dropna=False)
    pcts   = counts / counts.sum() * 100

    fig_bar = px.bar(
        x=counts.index.astype(str), y=counts.values,
        labels={"x": col, "y": "Count"}, title=f"Count — {col}",
        text=counts.values,
    )
    fig_bar.update_layout(xaxis_tickangle=-40)
    fig_bar.show()

    fig_pie = px.pie(
        names=pcts.index.astype(str), values=pcts.values,
        title=f"Distribution — {col}", hole=0.0,
    )
    fig_pie.update_traces(textposition="inside", textinfo="percent+label")
    fig_pie.show()

## 3 · Bivariate analysis

### 3.1 Correlation matrix

In [ ]:
num_df = l_df.select_dtypes(include=["int64", "float64"])
corr   = num_df.corr()

fig = px.imshow(
    corr, text_auto=False, color_continuous_scale="RdBu_r",
    aspect="auto", title="Correlation matrix — numerical features",
)
fig.update_layout(height=900)
fig.show()

### 3.2 Price vs key numerical variables

In [ ]:
price_cap = l_df["price"].quantile(0.99)
df_plot   = l_df[l_df["price"] <= price_cap].copy()

for x_col in ["review_scores_rating", "accommodates", "number_of_reviews"]:
    r = l_df["price"].corr(l_df[x_col])
    fig = px.scatter(
        df_plot, x=x_col, y="price", trendline="ols", opacity=0.4,
        title=f"Price vs {x_col}   (Pearson r = {r:.3f})",
    )
    fig.show()

### 3.3 Price by room type

In [ ]:
display(l_df.groupby("room_type")["price"]
        .agg(["mean", "median", "std", "count"]))

fig = px.box(
    l_df, x="room_type", y="price",
    title="Price distribution by room type", points=False,
)
fig.show()

median_prices = l_df.groupby("room_type")["price"].median().reset_index()
fig2 = px.bar(median_prices, x="room_type", y="price",
              title="Median price by room type",
              labels={"price": "Median price (€)"})
fig2.show()

groups = [g["price"].dropna() for _, g in l_df.groupby("room_type")]
f, p = f_oneway(*groups)
print(f"ANOVA: F = {f:.4f},  p = {p:.2e}")

### 3.4 Reviews by neighbourhood (top 15)

In [ ]:
top_neigh = l_df["neighbourhood_cleansed"].value_counts().head(15).index
l_top     = l_df[l_df["neighbourhood_cleansed"].isin(top_neigh)]

display(l_top.groupby("neighbourhood_cleansed")["number_of_reviews"]
        .agg(["count", "mean", "median", "std"])
        .sort_values("mean", ascending=False))

fig = px.box(
    l_top, x="neighbourhood_cleansed", y="number_of_reviews",
    title="Number of reviews by neighbourhood (top 15)", points=False,
)
fig.update_layout(width=1100, height=500, xaxis_tickangle=45)
fig.show()

groups = [g["number_of_reviews"].dropna() for _, g in l_top.groupby("neighbourhood_cleansed")]
f, p = f_oneway(*groups)
print(f"ANOVA: F = {f:.4f},  p = {p:.2e}")

### 3.5 Superhost vs non-superhost

In [ ]:
metrics = ["price", "number_of_reviews", "review_scores_rating"]
display(l_df.groupby("host_is_superhost")[metrics]
        .agg(["mean", "median", "std"]))

for m in metrics:
    fig = px.box(l_df, x="host_is_superhost", y=m,
                 title=f"{m} — Superhost vs Non-Superhost", points=False)
    fig.show()

fig_resp = px.histogram(
    l_df, x="host_response_time", color="host_is_superhost",
    barmode="group", title="Response time by superhost status",
    labels={"host_is_superhost": "Superhost"},
)
fig_resp.show()

sh  = l_df[l_df["host_is_superhost"]]["price"].dropna()
nsh = l_df[~l_df["host_is_superhost"]]["price"].dropna()
t, p = ttest_ind(sh, nsh, equal_var=False)
print(f"Welch t-test on price: t = {t:.4f},  p = {p:.4f}")

## 4 · Multivariate analysis

### 4.1 Price category × occupancy & revenue

In [ ]:
cat_order = ["low", "medium", "high", "very_high"]

if "estimated_occupancy_l365d" in l_df.columns:
    display(l_df.groupby("price_cat", observed=True)["estimated_occupancy_l365d"]
            .median().reindex(cat_order))
    fig = px.box(l_df, x="price_cat", y="estimated_occupancy_l365d",
                 category_orders={"price_cat": cat_order},
                 title="Estimated occupancy (365 d) by price category",
                 points=False)
    fig.show()

if "estimated_revenue_l365d" in l_df.columns:
    rev_df = l_df.dropna(subset=["estimated_revenue_l365d"])
    display(rev_df.groupby("price_cat", observed=True)["estimated_revenue_l365d"]
            .median().reindex(cat_order))
    fig = px.box(rev_df, x="price_cat", y="estimated_revenue_l365d",
                 category_orders={"price_cat": cat_order},
                 title="Estimated revenue (365 d) by price category",
                 points=False)
    fig.show()

### 4.2 Property type × neighbourhood

In [ ]:
top_neigh = l_df["neighbourhood_cleansed"].value_counts().head(15).index
l_sub = l_df[l_df["neighbourhood_cleansed"].isin(top_neigh)]

# Stacked proportional bar
ct = pd.crosstab(l_sub["neighbourhood_cleansed"], l_sub["property_type_std"])
ct_prop = ct.div(ct.sum(axis=1), axis=0)

fig = px.bar(
    ct_prop.reset_index().melt(
        id_vars="neighbourhood_cleansed",
        var_name="property_type_std", value_name="proportion",
    ),
    x="neighbourhood_cleansed", y="proportion",
    color="property_type_std",
    title="Property type mix by neighbourhood",
    barmode="stack",
)
fig.update_layout(height=600, width=1100, xaxis_tickangle=45)
fig.show()

# Room type heatmap across district groups
ct2 = pd.crosstab(l_df["neighbourhood_group_cleansed"], l_df["room_type"])
fig2 = px.imshow(
    ct2[["Entire home/apt", "Private room"]],
    title="Entire home vs private room by district",
    text_auto=True, aspect="auto",
)
fig2.update_layout(height=500, width=700)
fig2.show()

### 4.3 3-D scatter: review score × price × review volume

In [ ]:
price_cap = l_df["price"].quantile(0.99)
df_no_out = l_df[l_df["price"] <= price_cap].copy()

fig = px.scatter_3d(
    df_no_out,
    x="review_scores_rating", y="price", z="number_of_reviews",
    color="property_type_std", size="number_of_reviews", opacity=0.6,
    title="Review score × Price × Review volume (outliers removed)",
    labels={"review_scores_rating": "Review Score",
            "price": "Price (€)", "number_of_reviews": "# Reviews"},
)
fig.update_layout(height=750, width=1000)
fig.show()

### 4.4 K-means market segmentation

In [ ]:
feat_cols = ["price", "availability_365", "review_scores_rating", "number_of_reviews"]
features  = l_df[feat_cols].dropna().copy()
scaled    = StandardScaler().fit_transform(features)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
features["cluster"] = kmeans.fit_predict(scaled)

fig = px.scatter(
    features, x="price", y="availability_365", color="cluster",
    title="Market segments — Price × Availability (k = 4)",
)
fig.show()

l_df = l_df.merge(features[["cluster"]], left_index=True, right_index=True, how="left")

segment_summary = l_df.groupby("cluster").agg(
    count          = ("price", "count"),
    median_price   = ("price", "median"),
    median_avail   = ("availability_365", "median"),
    median_reviews = ("number_of_reviews", "median"),
    median_rating  = ("review_scores_rating", "median"),
)
display(segment_summary)

## 5 · Key findings

_Fill in the most important insights discovered in this notebook._